# JS09 — Tugas Lab 2: ANNOY vs FAISS vs HNSW pada Dataset Spotify

**Praktikum Pembelajaran Mesin — Politeknik Negeri Malang**

| | |
|---|---|
| Nama | TBD |
| NIM | TBD |
| Kelas | TBD |
| Tanggal | TBD |

**Tujuan:** mencari **track terdekat** pada dataset Spotify (fitur audio numerik, distandarisasi) memakai exact NN, ANNOY, FAISS, dan HNSW, lalu membandingkan hasil, recall, dan waktu query.

**Prasyarat:** dataset Kaggle *Spotify Songs with Attributes and Lyrics* (diunduh/unggah manual — lihat Langkah 0) dan library `annoy`, `faiss-cpu`, `hnswlib`, `scikit-learn`.

> Jalankan di **Google Colab**: `Runtime` ▸ `Run all`. Notebook ini cukup memakai CPU, tidak perlu GPU.


# Tugas Lab 2 — ANN pada Dataset Spotify

Modul: `JS09 › Tugas Lab 2`

**Instruksi modul:**

1. Unduh dan muat dataset sekunder berukuran besar (**Micro Spotify**) dari Kaggle — pilih **dataset pertama** dari dua file yang tersedia.
2. Pilih hanya **fitur numerik**, lalu normalisasi dengan **StandardScaler**.
3. Lakukan pencarian **track terdekat** dan bandingkan hasilnya.
4. Bandingkan **exact NN, ANNOY, FAISS, dan HNSW**.

Sumber data: <https://www.kaggle.com/datasets/bwandowando/spotify-songs-with-attributes-and-lyrics/data>

**Fitur yang dipakai** (sesuai boilerplate modul): `danceability`, `energy`, `loudness`, `speechiness`, `acousticness`, `instrumentalness`, `liveness`, `valence`, `tempo`. Kolom `track_name`/`track_artist`/`track_popularity` hanya dipakai untuk menampilkan hasil, bukan untuk menghitung jarak.

> **Penting:** dataset ini berisi lebih dari satu file (atribut audio dan lirik). Pilih file **atribut audio** (yang punya kolom `danceability`, `energy`, dst.) — biasanya `spotify_songs.csv`. Notebook hanya membaca kolom yang dibutuhkan, sehingga file besar tetap aman dimuat.


### Persiapan — Install & Import Library

Colab belum menyertakan `annoy`, `faiss`, dan `hnswlib`. Cell berikut memasangnya hanya bila belum tersedia.

In [ ]:
# Install library bila belum tersedia (Colab)
try:
    import annoy  # noqa: F401
except ImportError:
    !pip install -q annoy

try:
    import faiss  # noqa: F401
except ImportError:
    !pip install -q faiss-cpu

try:
    import hnswlib  # noqa: F401
except ImportError:
    !pip install -q hnswlib

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Langkah 0 — Menyiapkan Dataset

Kaggle memerlukan login untuk mengunduh. Pilihan cara mendapatkan file:

1. **Manual (paling mudah):** buka halaman Kaggle → `Download` → ekstrak → unggah file CSV atribut ke Colab (`Files` ▸ *Upload*) atau ke folder kerja notebook ini.
2. **Kaggle CLI** (butuh `kaggle.json` dari *Account ▸ Create New API Token*):
   ```bash
   !pip install -q kaggle
   !kaggle datasets download -d bwandowando/spotify-songs-with-attributes-and-lyrics
   !unzip -o spotify-songs-with-attributes-and-lyrics.zip
   ```

Cell di bawah mencari file `*spotify*.csv` di folder kerja; bila tidak ada, Colab akan meminta Anda mengunggahnya.

In [ ]:
from pathlib import Path

NAMA_FILE = "spotify_songs.csv"
KANDIDAT = [NAMA_FILE] + sorted(p.name for p in Path(".").glob("*spotify*.csv"))
DATASET_PATH = next((p for p in KANDIDAT if Path(p).exists()), None)

if DATASET_PATH is None:
    try:                       # khusus Google Colab
        from google.colab import files
        print("Unggah file CSV dataset Spotify (file atribut audio):")
        uploaded = files.upload()
        DATASET_PATH = next(iter(uploaded), None) if uploaded else None
    except ImportError:
        pass

if DATASET_PATH is None:
    print("Dataset belum ditemukan. Unduh dari Kaggle lalu letakkan CSV-nya di folder kerja notebook ini:")
    print("https://www.kaggle.com/datasets/bwandowando/spotify-songs-with-attributes-and-lyrics/data")
    print()
    print("Alternatif di Colab (butuh kaggle.json):")
    print("  !pip install -q kaggle")
    print("  !kaggle datasets download -d bwandowando/spotify-songs-with-attributes-and-lyrics")
    print("  !unzip -o spotify-songs-with-attributes-and-lyrics.zip")

print("DATASET_PATH =", DATASET_PATH)

## Langkah 1 — Load Dataset

Cell ini membaca **header** lebih dulu, lalu hanya memuat kolom yang dipakai. Dengan begitu kolom lirik yang besar tidak ikut masuk ke memori.

In [ ]:
import pandas as pd

FITUR = ['danceability', 'energy', 'loudness', 'speechiness',
         'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
KOLOM_META = ['track_id', 'track_name', 'track_artist', 'track_popularity']

assert DATASET_PATH is not None, "Isi DATASET_PATH lebih dulu (lihat Langkah 0)."

semua_kolom = pd.read_csv(DATASET_PATH, nrows=0).columns.tolist()
print("Jumlah kolom di file:", len(semua_kolom))
print(semua_kolom)

dipakai = [k for k in FITUR + KOLOM_META if k in semua_kolom]
hilang = [k for k in FITUR if k not in semua_kolom]
if hilang:
    print("PERINGATAN: fitur tidak ditemukan di file ->", hilang)
assert dipakai, "Tidak ada kolom yang cocok — pastikan file yang diunggah adalah dataset atribut audio Spotify."
print("Kolom yang dimuat:", dipakai)

df = pd.read_csv(DATASET_PATH, usecols=dipakai)
print("Shape:", df.shape)
df.head()

## Langkah 2 — Bersihkan Data

Baris yang fitur numeriknya kosong dibuang, karena jarak Euclidean tidak bisa dihitung pada nilai `NaN`.

In [ ]:
fitur_tersedia = [k for k in FITUR if k in df.columns]
assert len(fitur_tersedia) >= 2, "Terlalu sedikit fitur numerik yang tersedia."

# Pastikan kolom fitur benar-benar numerik (nilai aneh dikonversi menjadi NaN)
for kol in fitur_tersedia:
    df[kol] = pd.to_numeric(df[kol], errors='coerce')

df = df.dropna(subset=fitur_tersedia).reset_index(drop=True)
print("Shape setelah dropna:", df.shape)
print("Jumlah track unik:", df['track_id'].nunique() if 'track_id' in df.columns else len(df))
df[fitur_tersedia].describe().T

### [tambahan] Batasi jumlah baris (opsional)

Dataset versi terbaru bisa berisi ratusan ribu track. ANN tetap jalan, tetapi HNSW memerlukan waktu build lebih lama. Isi `N_MAKS` bila ingin mempercepat; `None` berarti semua baris dipakai.

In [ ]:
# [tambahan] Batasi jumlah baris (None = semua baris)
N_MAKS = None          # contoh: 100_000

if N_MAKS is not None and len(df) > N_MAKS:
    df = df.sample(N_MAKS, random_state=42).reset_index(drop=True)

print("Jumlah track yang dipakai:", len(df))

## Langkah 3 — Pilih Fitur Numerik & StandardScaler

Standardisasi wajib di sini: `tempo` bernilai puluhan–ratusan, `loudness` bernilai negatif (dB), sedangkan fitur lain berada di 0–1. Tanpa scaling, jarak Euclidean akan didominasi `tempo` dan `loudness`.

In [ ]:
from sklearn.preprocessing import StandardScaler

X = df[fitur_tersedia].values.astype('float32')

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype('float32')

k = 10  # jumlah nearest neighbors
print("Shape X_scaled:", X_scaled.shape, "| dtype:", X_scaled.dtype)

## Langkah 4 — Pilih Track Query

Isi `QUERY_INDEX` atau `QUERY_TRACK`. Track query otomatis dikeluarkan dari daftar tetangga (jarak 0 ke dirinya sendiri tidak ditampilkan).

In [ ]:
QUERY_INDEX = 0          # indeks baris track query
QUERY_TRACK = None       # contoh: "Shape of You" -> dicari berdasarkan judul

if QUERY_TRACK and 'track_name' in df.columns:
    cocok = df.index[df['track_name'].str.contains(QUERY_TRACK, case=False, na=False)]
    if len(cocok):
        QUERY_INDEX = int(cocok[0])
    else:
        print("Track tidak ditemukan, memakai QUERY_INDEX =", QUERY_INDEX)

xq = X_scaled[[QUERY_INDEX]]

judul = df.loc[QUERY_INDEX, 'track_name'] if 'track_name' in df.columns else f"baris {QUERY_INDEX}"
artis = df.loc[QUERY_INDEX, 'track_artist'] if 'track_artist' in df.columns else "-"
print("Query:", judul, "—", artis)
print("Index:", QUERY_INDEX)
xq

## Langkah 5 — Exact NN (Baseline)

`NearestNeighbors` dengan `algorithm='brute'` menghitung jarak Euclidean ke **seluruh** track. Jumlah tetangga diminta `k + 1` agar track query sendiri bisa dibuang.

In [ ]:
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=k + 1, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)

start = time.time()
dist_raw, idx_raw = nn.kneighbors(xq)
t_exact = (time.time() - start) * 1000

pasangan = [(int(i), float(d)) for i, d in zip(idx_raw[0], dist_raw[0]) if int(i) != QUERY_INDEX][:k]
idx_exact = [i for i, _ in pasangan]
dist_exact = [d for _, d in pasangan]

print("Exact NN (sklearn brute):", idx_exact)
print("Jarak Euclidean:", [round(d, 4) for d in dist_exact])
print(f"Waktu query: {t_exact:.4f} ms")

## Langkah 6 — ANNOY

ANNOY membangun 50 pohon; `search_k` menentukan berapa banyak kandidat diperiksa saat query (semakin besar = semakin akurat, semakin lambat).

In [ ]:
from annoy import AnnoyIndex

N_TREES = 50
SEARCH_K = 2000

ann = AnnoyIndex(X_scaled.shape[1], 'euclidean')
for i, v in enumerate(X_scaled):
    ann.add_item(i, v)
t0 = time.time(); ann.build(N_TREES); t_build_ann = time.time() - t0

start = time.time()
kandidat_ann = ann.get_nns_by_vector(xq[0], k + 1, search_k=SEARCH_K)
t_q_ann = (time.time() - start) * 1000

idx_ann = [int(i) for i in kandidat_ann if int(i) != QUERY_INDEX][:k]
print("ANNOY NN:", idx_ann)
print(f"Build: {t_build_ann:.2f} s | Query: {t_q_ann:.4f} ms")

## Langkah 7 — FAISS

`IndexIVFFlat` membagi ruang vektor menjadi `NLIST` klaster, lalu saat query hanya `nprobe` klaster terdekat yang diperiksa. Ini versi *approximate* dari FAISS; `IndexFlatL2` (exact) dipakai nanti sebagai ground truth.

In [ ]:
import faiss

NLIST = 100
NPROBE = 10
d = X_scaled.shape[1]

quantizer = faiss.IndexFlatL2(d)
ivf = faiss.IndexIVFFlat(quantizer, d, NLIST, faiss.METRIC_L2)

t0 = time.time(); ivf.train(X_scaled); ivf.add(X_scaled); t_build_ivf = time.time() - t0
ivf.nprobe = NPROBE

start = time.time()
D_ivf, I_ivf = ivf.search(xq, k + 1)
t_q_ivf = (time.time() - start) * 1000

idx_ivf = [int(i) for i in I_ivf[0] if int(i) != QUERY_INDEX][:k]
print("FAISS IVF NN:", idx_ivf)
print(f"Build: {t_build_ivf:.2f} s | Query: {t_q_ivf:.4f} ms")

## Langkah 8 — HNSW

HNSW membangun graf bertingkat. `ef` saat query mengatur *trade-off* akurasi vs kecepatan; `M` dan `ef_construction` menentukan kualitas graf saat build.

In [ ]:
import hnswlib

M_HNSW = 16
EF_CONSTRUCTION = 200
EF_SEARCH = 100

hnsw = hnswlib.Index(space='l2', dim=d)
hnsw.init_index(max_elements=X_scaled.shape[0], ef_construction=EF_CONSTRUCTION, M=M_HNSW)

t0 = time.time(); hnsw.add_items(X_scaled); t_build_hnsw = time.time() - t0
hnsw.set_ef(EF_SEARCH)

start = time.time()
labels_hnsw, jarak_hnsw = hnsw.knn_query(xq, k=k + 1)
t_q_hnsw = (time.time() - start) * 1000

idx_hnsw = [int(i) for i in labels_hnsw[0] if int(i) != QUERY_INDEX][:k]
print("HNSW NN:", idx_hnsw)
print(f"Build: {t_build_hnsw:.2f} s | Query: {t_q_hnsw:.4f} ms")

## Langkah 9 — Bandingkan Track Terdekat

Tabel di bawah menampilkan 10 track terdekat menurut setiap metode. Perhatikan apakah isinya sama dan bagaimana urutannya.

In [ ]:
import numpy as np

def tabel_tetangga(idx, metode):
    idx = list(idx)
    judul = df['track_name'].values[idx] if 'track_name' in df.columns else np.array(idx).astype(str)
    artis = df['track_artist'].values[idx] if 'track_artist' in df.columns else np.array(['-'] * len(idx))
    pop = df['track_popularity'].values[idx] if 'track_popularity' in df.columns else np.full(len(idx), np.nan)
    return pd.DataFrame({
        'metode': metode,
        'peringkat': range(1, len(idx) + 1),
        'track_name': judul,
        'track_artist': artis,
        'popularity': pop,
    })

banding = pd.concat([
    tabel_tetangga(idx_exact, 'Exact (sklearn brute)'),
    tabel_tetangga(idx_ann, 'ANNOY'),
    tabel_tetangga(idx_ivf, 'FAISS IVF'),
    tabel_tetangga(idx_hnsw, 'HNSW'),
], ignore_index=True)

banding

## Langkah 10 — Recall & Waktu Query (satu query)

Recall@10 = jumlah track yang muncul di hasil ANN **dan** hasil exact, dibagi 10.

In [ ]:
def recall(pred, gt, k=k):
    return len(set(list(pred)[:k]) & set(list(gt)[:k])) / k

gt = list(idx_exact)

tabel_tugas2 = pd.DataFrame([
    {'metode': 'Exact (sklearn brute)', 'recall@10': recall(gt, gt), 'query (ms)': t_exact},
    {'metode': 'ANNOY', 'recall@10': recall(idx_ann, gt), 'query (ms)': t_q_ann},
    {'metode': 'FAISS IVF', 'recall@10': recall(idx_ivf, gt), 'query (ms)': t_q_ivf},
    {'metode': 'HNSW', 'recall@10': recall(idx_hnsw, gt), 'query (ms)': t_q_hnsw},
]).round(4)

tabel_tugas2

### [tambahan] Ukur ulang dengan 200 query + QPS

Waktu satu query sangat bising. Cell ini memakai 200 track acak sebagai query, `IndexFlatL2` sebagai ground truth, dan menghitung QPS (`1000 / ms per query`).

In [ ]:
# [tambahan] 200 query: recall@10, ms/query, QPS
rng = np.random.default_rng(42)
qidx = rng.choice(len(X_scaled), size=min(200, len(X_scaled)), replace=False)
Q = X_scaled[qidx]

flat = faiss.IndexFlatL2(d)
t0 = time.time(); flat.add(X_scaled); t_build_flat = time.time() - t0

start = time.time(); _, I_gt = flat.search(Q, k + 1)
t_q_flat = (time.time() - start) * 1000 / len(Q)

start = time.time()
I_ann_many = [ann.get_nns_by_vector(q, k + 1, search_k=SEARCH_K) for q in Q]
t_q_ann_many = (time.time() - start) * 1000 / len(Q)

start = time.time(); _, I_ivf_many = ivf.search(Q, k + 1)
t_q_ivf_many = (time.time() - start) * 1000 / len(Q)

start = time.time(); I_hnsw_many, _ = hnsw.knn_query(Q, k=k + 1)
t_q_hnsw_many = (time.time() - start) * 1000 / len(Q)

def tanpa_self(I, qidx, k=k):
    hasil = []
    for baris, diri in zip(I, qidx):
        hasil.append([int(i) for i in baris if int(i) != int(diri)][:k])
    return hasil

gt_many = tanpa_self(I_gt, qidx)

def rerata_recall(pred):
    return float(np.mean([recall(p, g) for p, g in zip(tanpa_self(pred, qidx), gt_many)]))

rekap_tugas2 = pd.DataFrame([
    {'metode': 'FAISS Flat (exact)', 'recall@10': 1.0, 'ms/query': t_q_flat},
    {'metode': 'ANNOY', 'recall@10': rerata_recall(I_ann_many), 'ms/query': t_q_ann_many},
    {'metode': 'FAISS IVF', 'recall@10': rerata_recall(I_ivf_many), 'ms/query': t_q_ivf_many},
    {'metode': 'HNSW', 'recall@10': rerata_recall(I_hnsw_many), 'ms/query': t_q_hnsw_many},
])
rekap_tugas2['QPS'] = (1000 / rekap_tugas2['ms/query']).round(1)
rekap_tugas2 = rekap_tugas2.round(4)
rekap_tugas2

**Catatan:** `t_q_flat` pada tabel ini adalah waktu `IndexFlatL2` (exact) pada 200 query, sedangkan `t_exact` di tabel sebelumnya berasal dari sklearn pada satu query — keduanya sama-sama exact, tetapi tidak identik.

### [tambahan] Pengaruh parameter terhadap recall & waktu

In [ ]:
# [tambahan] Scan parameter pada 100 query
Q_sub = Q[:100]
qi_sub = qidx[:100]
gt_sub = tanpa_self(flat.search(Q_sub, k + 1)[1], qi_sub)

def rerata(pred):
    return float(np.mean([recall(p, g) for p, g in zip(pred, gt_sub)]))

rows = []

for search_k in [100, 500, 2000]:
    start = time.time()
    pred = [ann.get_nns_by_vector(q, k + 1, search_k=search_k) for q in Q_sub]
    ms = (time.time() - start) * 1000 / len(Q_sub)
    rows.append({'algoritma': 'ANNOY', 'parameter': f'search_k={search_k}',
                 'recall@10': rerata(tanpa_self(pred, qi_sub)), 'ms/query': ms})

for nprobe in [1, 10, 50]:
    ivf.nprobe = nprobe
    start = time.time(); _, I = ivf.search(Q_sub, k + 1)
    ms = (time.time() - start) * 1000 / len(Q_sub)
    rows.append({'algoritma': 'FAISS IVF', 'parameter': f'nprobe={nprobe}',
                 'recall@10': rerata(tanpa_self(I, qi_sub)), 'ms/query': ms})

for ef in [20, 50, 100, 200]:
    hnsw.set_ef(ef)
    start = time.time(); I, _ = hnsw.knn_query(Q_sub, k=k + 1)
    ms = (time.time() - start) * 1000 / len(Q_sub)
    rows.append({'algoritma': 'HNSW', 'parameter': f'ef={ef}',
                 'recall@10': rerata(tanpa_self(I, qi_sub)), 'ms/query': ms})

tugas2_sweep = pd.DataFrame(rows).round(4)
tugas2_sweep

In [ ]:
# [tambahan] Grafik recall vs waktu dari hasil scan parameter
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
for nama, penanda in zip(['ANNOY', 'FAISS IVF', 'HNSW'], ['o', 's', '^']):
    sub = tugas2_sweep[tugas2_sweep['algoritma'] == nama]
    ax.plot(sub['ms/query'], sub['recall@10'], marker=penanda, label=nama)

ax.set_xscale('log')
ax.set_xlabel('Waktu query (ms/query, log)')
ax.set_ylabel('Recall@10')
ax.set_title('Spotify: trade-off recall vs waktu query')
ax.legend()
ax.grid(True, ls='--', alpha=0.6)
plt.tight_layout()
plt.show()

**Analisis:** TBD

*(Jelaskan: (1) apakah 10 track terdekat dari ANNOY/FAISS/HNSW sama dengan exact, dan mana yang paling berbeda; (2) nilai recall@10 masing-masing pada tabel 200 query; (3) mengapa standardisasi penting untuk dataset ini — sebutkan rentang `tempo`/`loudness` vs fitur 0–1; (4) interpretasi grafik scan parameter: parameter mana yang paling berpengaruh terhadap recall; (5) metode mana yang Anda pilih untuk aplikasi rekomendasi lagu, beserta alasannya.)*

---
## Kesimpulan Tugas Lab 2

**Track terdekat hasil exact vs ANNOY/FAISS/HNSW (sama atau berbeda):** TBD

**Recall@10 tiap metode:** TBD

**Perbandingan waktu build & query:** TBD

**Pengaruh standardisasi & parameter:** TBD

**Rekomendasi metode untuk pencarian lagu serupa:** TBD